# Consolidação de Bancos de Dados para Treinamento de YOLO

Este notebook executa o script de consolidação para juntar imagens de múltiplos bancos de dados do Roboflow em um único dataset padronizado, unificando classes para `lixo` (ID 0) e gerando anotações visuais para validação manual.

In [ ]:
# Adiciona o diretório raiz ao path para importação do módulo src
import sys
import os
sys.path.append(os.path.abspath('..'))

## 1. Executar a Consolidação

Importa e executa a lógica de consolidação contida em `src/data/consolidate.py`.

In [ ]:
from src.data.consolidate import run_consolidation
run_consolidation(base_dir='..')

## 2. Estatísticas do Dataset Consolidado

Carrega o mapeamento (`mapping.json`) e a tabela de rastreabilidade (`traceability.csv`) para analisar o dataset gerado.

In [ ]:
import pandas as pd
import json

# Carrega mapping
with open('../data/metadata/mapping.json', 'r', encoding='utf-8') as f:
    mapping = json.load(f)

# Exibe a tabela de mapeamento de datasets
df_mapping = pd.DataFrame.from_dict(mapping, orient='index')
display(df_mapping[['alias', 'original_folder', 'dataset_name', 'roboflow_url', 'classes']])

In [ ]:
# Carrega a tabela de rastreabilidade
df_trace = pd.read_csv('../data/metadata/traceability.csv')

print(f"Total de imagens consolidadas: {len(df_trace)}")
print("\nDistribuição por Dataset de Origem:")
print(df_trace['original_dataset'].value_counts())

print("\nDistribuição de Splits Originais:")
print(df_trace['original_split'].value_counts())

## 3. Visualização de Amostras Anotadas (QA)

Plota algumas imagens aleatórias com suas bounding boxes desenhadas da pasta `data/visual` para inspeção visual rápida.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random
import glob

visual_images = glob.glob('../data/visual/*.jpg') + glob.glob('../data/visual/*.png')

if visual_images:
    # Seleciona 4 imagens aleatórias
    samples = random.sample(visual_images, min(4, len(visual_images)))
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.ravel()
    
    for idx, img_path in enumerate(samples):
        img = cv2.imread(img_path)
        # OpenCV carrega em BGR, convertemos para RGB para exibir no Matplotlib
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(img_rgb)
        axes[idx].set_title(os.path.basename(img_path))
        axes[idx].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma imagem encontrada na pasta data/visual/")